# Fase 6 — Deployment | CardioRisk
**IBM Data Science Professional Certificate · CRISP-DM**

EWS (Early Warning Score) alineado con Business Understanding (Fase 1):
- Bajo Riesgo: p < 0.30 → Monitoreo estándar
- Riesgo Medio: 0.30 ≤ p ≤ 0.65 → Vigilancia intensiva
- Alto Riesgo: p > 0.65 → Alerta inmediata UCI

In [ ]:
# BLOQUE 0 — Instalaciones e imports
!pip install -q kagglehub scikit-learn imbalanced-learn joblib
import os, warnings, joblib
import pandas as pd
import numpy as np
import matplotlib, matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (recall_score, precision_score, f1_score, roc_auc_score, accuracy_score, confusion_matrix, roc_curve)
from imblearn.over_sampling import SMOTE
# Umbrales EWS — definidos en Fase 1 Business Understanding
EWS_BAJO  = 0.30
EWS_MEDIO = 0.65
COLOR_BG='#0a0f1a'; COLOR_AX='#0d1526'; COLOR_TEXT='#e2e8f0'; COLOR_GRID='#1a2c3d'
matplotlib.rcParams.update({'figure.facecolor':COLOR_BG,'axes.facecolor':COLOR_AX,'axes.edgecolor':COLOR_GRID,'axes.labelcolor':COLOR_TEXT,'xtick.color':'#7a8fa8','ytick.color':'#7a8fa8','grid.color':COLOR_GRID,'text.color':COLOR_TEXT})
print('✅ Entorno listo')
print(f'EWS umbrales: BAJO<{EWS_BAJO} | MEDIO {EWS_BAJO}-{EWS_MEDIO} | ALTO>{EWS_MEDIO}')

In [ ]:
# BLOQUE 1 — Pipeline completo final (18 features: 11 orig + 3 engineered + 4 OHE)
print('='*60); print('PIPELINE FASE 6 — DEPLOYMENT CARDIORISK'); print('='*60)
try:
    import kagglehub
    path = kagglehub.dataset_download('jocelyndumlao/cardiovascular-disease-dataset')
    csv_path = next(os.path.join(r,f) for r,_,fs in os.walk(path) for f in fs if f.endswith('.csv'))
    df = pd.read_csv(csv_path); print(f'✅ KaggleHub: {csv_path}')
except Exception:
    df = pd.read_csv('/content/cardiovascular_disease_dataset.csv'); print('✅ CSV local')
df.columns = df.columns.str.lower().str.strip(); df = df.dropna()
TARGET='target'; CATEGORICAL=['gender','chestpain','restingrelectro']
# Feature Engineering — 3 nuevas variables (Fase 3)
df['slope_x_oldpeak'] = df['slope'] * df['oldpeak']
df['log_oldpeak']     = np.log1p(df['oldpeak'])
df['vessels_bin']     = (df['noofmajorvessels'] >= 2).astype(int)
df_enc = pd.get_dummies(df, columns=CATEGORICAL, drop_first=False)
feature_cols = ['age','gender_0','gender_1','restingbp','serumcholestrol','fastingbloodsugar',
                'restingrelectro_0','restingrelectro_1','restingrelectro_2',
                'maxheartrate','exerciseangia','oldpeak','slope','noofmajorvessels',
                'slope_x_oldpeak','log_oldpeak','vessels_bin',
                'chestpain_0','chestpain_1','chestpain_2','chestpain_3']
# Filtrar solo las que existen en df_enc
feature_cols = [c for c in feature_cols if c in df_enc.columns]
print(f'✅ Features: {len(feature_cols)} | patientid excluido')
X = df_enc[feature_cols]; y = df_enc[TARGET]
X_temp,X_test,y_temp,y_test = train_test_split(X,y,test_size=0.15,stratify=y,random_state=42)
X_train,X_val,y_train,y_val = train_test_split(X_temp,y_temp,test_size=0.1765,stratify=y_temp,random_state=42)
scaler=StandardScaler(); X_train_sc=scaler.fit_transform(X_train)
X_val_sc=scaler.transform(X_val); X_test_sc=scaler.transform(X_test)
smote=SMOTE(random_state=42); X_train_sm,y_train_sm=smote.fit_resample(X_train_sc,y_train)
model=GradientBoostingClassifier(learning_rate=0.1,max_depth=5,n_estimators=200,random_state=42)
model.fit(X_train_sm,y_train_sm); print('✅ GradientBoostingClassifier entrenado')
joblib.dump(model,'/content/cardiorisk_model.pkl')
joblib.dump(scaler,'/content/cardiorisk_scaler.pkl')
joblib.dump(feature_cols,'/content/cardiorisk_features.pkl')
print('✅ Artefactos guardados')

In [ ]:
# BLOQUE 2 — Métricas TEST SET
y_pred=model.predict(X_test_sc); y_pred_proba=model.predict_proba(X_test_sc)[:,1]
recall=recall_score(y_test,y_pred); precision=precision_score(y_test,y_pred)
f1=f1_score(y_test,y_pred); auc=roc_auc_score(y_test,y_pred_proba)
accuracy=accuracy_score(y_test,y_pred); cm_matrix=confusion_matrix(y_test,y_pred)
tn,fp,fn,tp=cm_matrix.ravel()
print('='*60); print('EVALUACIÓN FINAL — TEST SET (EWS CardioRisk)'); print('='*60)
print(f'  Recall    : {recall:.4f}  ← KPI primario (Error II={fn/(tp+fn):.1%})')
print(f'  Precision : {precision:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  AUC-ROC   : {auc:.4f}')
print(f'  TP={tp} TN={tn} FP={fp} FN={fn}')
print(f'  KPI Recall>0.80:       {"✅" if recall>=0.80 else "❌"}')
print(f'  KPI AUC-ROC>0.85:      {"✅" if auc>0.85 else "❌"}')
print(f'  KPI Error tipo II<5%:  {"✅" if fn/(tp+fn)<0.05 else "❌"}')

In [ ]:
# BLOQUE 3 — EWS: función deployable
def predecir_riesgo_cardiovascular(paciente: dict, modelo=model, sc=scaler, cols=feature_cols) -> dict:
    df_p = pd.DataFrame([paciente])
    df_p.columns = df_p.columns.str.lower().str.strip()
    # Feature engineering
    if 'slope' in df_p.columns and 'oldpeak' in df_p.columns:
        df_p['slope_x_oldpeak'] = df_p['slope'] * df_p['oldpeak']
        df_p['log_oldpeak']     = np.log1p(df_p['oldpeak'])
    if 'noofmajorvessels' in df_p.columns:
        df_p['vessels_bin'] = (df_p['noofmajorvessels'] >= 2).astype(int)
    # OHE manual
    for cat in ['gender','chestpain','restingrelectro']:
        if cat in df_p.columns:
            val = int(df_p[cat].iloc[0])
            unique_vals = sorted([int(str(c).replace(cat+'_','')) for c in cols if c.startswith(cat+'_')])
            for v in unique_vals:
                df_p[f'{cat}_{v}'] = 1 if val == v else 0
            df_p = df_p.drop(columns=[cat])
    for col in cols:
        if col not in df_p.columns: df_p[col] = 0
    df_p = df_p[cols]
    proba = float(modelo.predict_proba(sc.transform(df_p.values))[0,1])
    pred  = 1 if proba >= 0.5 else 0
    if proba < EWS_BAJO:
        nivel,accion = 'Bajo Riesgo','Monitoreo estándar.'
    elif proba <= EWS_MEDIO:
        nivel,accion = 'Riesgo Medio','Vigilancia intensiva. Observación 6-12h. Biomarcadores.'
    else:
        nivel,accion = 'Alto Riesgo','ALERTA INMEDIATA UCI. Protocolo SCA. Cardiólogo STAT.'
    return {'riesgo':pred,'probabilidad':round(proba,4),'ews_nivel':nivel,'accion_clinica':accion}
# 3 casos de prueba
casos=[('Mujer 35a saludable',{'age':35,'restingbp':118,'serumcholestrol':175,'maxheartrate':155,'oldpeak':0.3,'noofmajorvessels':0,'fastingbloodsugar':0,'exerciseangia':0,'slope':2,'gender':0,'chestpain':0,'restingrelectro':0}),('Hombre 58a riesgo medio',{'age':58,'restingbp':140,'serumcholestrol':245,'maxheartrate':130,'oldpeak':1.2,'noofmajorvessels':1,'fastingbloodsugar':1,'exerciseangia':0,'slope':1,'gender':1,'chestpain':2,'restingrelectro':1}),('Hombre 67a alto riesgo UCI',{'age':67,'restingbp':162,'serumcholestrol':285,'maxheartrate':108,'oldpeak':3.6,'noofmajorvessels':3,'fastingbloodsugar':1,'exerciseangia':1,'slope':0,'gender':1,'chestpain':3,'restingrelectro':2})]
for nombre,datos in casos:
    r=predecir_riesgo_cardiovascular(datos)
    print(f'\nPaciente: {nombre}')
    print(f'  EWS: {r["ews_nivel"]}  (p={r["probabilidad"]:.1%})')
    print(f'  Acción: {r["accion_clinica"]}')
print('\n✅ EWS deployable OK')

In [ ]:
# BLOQUE 5 — Reporte CRISP-DM completo
s1='═'*70; s2='─'*70
print(f'\n{s1}'); print(' '*20+'REPORTE FINAL CRISP-DM — CARDIORISK')
print(' '*20+'IBM Data Science Professional Certificate'); print(s1)
print(f'\n{s2}\n1. BUSINESS UNDERSTANDING\n{s2}')
print('  Obj. Negocio: Reducir mortalidad hospitalaria 20%, optimizar rotación UCI.')
print('  Obj. DS:      Clasificador con EWS auditable para estratificar riesgo isquémico.')
print('  KPIs:         Recall>80% | AUC-ROC>0.85 | Error tipo II<5%')
print(f'\n{s2}\n2. DATA UNDERSTANDING\n{s2}')
print(f'  Dataset: jocelyndumlao/cardiovascular-disease-dataset (KaggleHub)')
print(f'  N={len(df)} | 14 columnas originales | patientid excluido')
print(f'\n{s2}\n3. DATA PREPARATION\n{s2}')
print(f'  18 features: 11 orig + 3 engineered (slope_x_oldpeak, log_oldpeak, vessels_bin) + 4 OHE chestpain')
print('  Split 70/15/15 estratificado | Scaler fit SOLO en train | SMOTE SOLO en train')
print(f'\n{s2}\n4. MODELING\n{s2}')
print('  GradientBoosting ganador: learning_rate=0.1, max_depth=5, n_estimators=200')
print(f'\n{s2}\n5. EVALUATION — TEST SET\n{s2}')
print(f'  Recall={recall:.4f} | AUC={auc:.4f} | F1={f1:.4f} | FN={fn}')
print(f'\n{s2}\n6. DEPLOYMENT — EWS CardioRisk\n{s2}')
print(f'  Bajo Riesgo  p<{EWS_BAJO}  → Monitoreo estándar')
print(f'  Riesgo Medio {EWS_BAJO}-{EWS_MEDIO} → Vigilancia intensiva 6-12h')
print(f'  Alto Riesgo  p>{EWS_MEDIO} → ALERTA INMEDIATA UCI')
print(f'\n{s1}')
print(' '*20+'PROYECTO CARDIORISK COMPLETADO ✓')
print(s1)